In [1]:
import os

In [2]:
%pwd

'c:\\Users\\yshan\\Desktop\\text-summarizer-NLP\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\yshan\\Desktop\\text-summarizer-NLP'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
            self, 
            config_file_path = CONFIG_FILE_PATH,
            params_file_path = PARAMS_FILE_PATH
    ):
        self.config=read_yaml(config_file_path)
        self.params=read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion

        create_directories([config.root_dir])
        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from src.textSummarizer.logging import logger

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading file from :[{self.config.source_URL}] into :[{self.config.local_data_file}]")

            file_name, headers = request.urlretrieve(
                url=self.config.source_URL, 
                filename=self.config.local_data_file
            )

            logger.info(f"File :[{self.config.local_data_file}] has been downloaded successfully.")
        else:
            logger.info(f"File :[{self.config.local_data_file}] already exists. Skipping download.")

    def unzip_and_save(self):      
        logger.info(f"Unzipping file :[{self.config.local_data_file}] to dir :[{self.config.unzip_dir}]")
        upzip_dir=self.config.unzip_dir
        os.makedirs(upzip_dir, exist_ok=True)

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

        logger.info(f"File :[{self.config.local_data_file}] has been unzipped successfully.")

In [11]:
config=ConfigurationManager()
data_ingestion_config=config.get_data_ingestion_config()
data_ingestion=DataIngestion(config=data_ingestion_config)
data_ingestion.download_file()
data_ingestion.unzip_and_save()

[2026-04-14 14:22:10,001: INFO: common]: Directory created at: artifacts
[2026-04-14 14:22:10,002: INFO: common]: Directory created at: artifacts/data_ingestion
[2026-04-14 14:22:10,003: INFO: 874578660]: Downloading file from :[https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip] into :[artifacts/data_ingestion/summarizer-data.zip]
[2026-04-14 14:22:13,470: INFO: 874578660]: File :[artifacts/data_ingestion/summarizer-data.zip] has been downloaded successfully.
[2026-04-14 14:22:13,471: INFO: 874578660]: Unzipping file :[artifacts/data_ingestion/summarizer-data.zip] to dir :[artifacts/data_ingestion/summarizer-data]
[2026-04-14 14:22:13,561: INFO: 874578660]: File :[artifacts/data_ingestion/summarizer-data.zip] has been unzipped successfully.
